# 002 · Environment Variables and Platform Info

> **Standard Library › System & CLI** · Python 3.14

Programs need to know where they run: configuration often arrives through **environment variables**, and code sometimes has to adapt to the operating system or the CPU. Python spreads this across `os.environ`, a few `os` functions and the `platform` module — .NET puts it in `Environment` and `RuntimeInformation`.

**In this notebook you will learn:**

- reading environment variables safely with `os.environ` and `os.getenv`
- setting variables, and how that affects child processes
- why you should never print the whole environment
- CPU count, process id, OS and architecture with `os` and `platform`
- portable OS checks with `sys.platform`, and `os.sep` / `os.linesep`

## .NET mental model

| .NET | Python |
|---|---|
| `Environment.GetEnvironmentVariable("X")` | `os.environ.get("X")` / `os.getenv("X")` |
| `Environment.SetEnvironmentVariable("X", "v")` | `os.environ["X"] = "v"` |
| `Environment.GetEnvironmentVariables()` | `os.environ` (a live `dict`-like mapping) |
| `Environment.ProcessorCount` | `os.cpu_count()` / `os.process_cpu_count()` |
| `Environment.ProcessId` | `os.getpid()` |
| `OperatingSystem.IsWindows()` / `IsLinux()` / `IsMacOS()` | `sys.platform == "win32"` / `.startswith("linux")` / `== "darwin"` |
| `RuntimeInformation.OSDescription` | `platform.platform()` |
| `RuntimeInformation.ProcessArchitecture` | `platform.machine()` |
| `RuntimeInformation.FrameworkDescription` | `platform.python_version()` / `python_implementation()` |
| `Path.DirectorySeparatorChar` / `Path.PathSeparator` | `os.sep` / `os.pathsep` |
| `Environment.NewLine` | `os.linesep` |

## Reading environment variables

`os.environ` behaves like a `dict[str, str]`. Use `.get()` with a default when a variable is optional — indexing a missing key raises `KeyError`, unlike .NET's `GetEnvironmentVariable`, which returns `null`.

```csharp
// C#
string? mode = Environment.GetEnvironmentVariable("APP_MODE");      // null if missing
string level = Environment.GetEnvironmentVariable("LOG_LEVEL") ?? "INFO";
```

In [ ]:
import os

print("PATH" in os.environ)                          # True — set on every OS
print(os.environ.get("APP_MODE"))                    # None (C#: null)
print(os.environ.get("LOG_LEVEL", "INFO"))           # default when missing
print(os.getenv("LOG_LEVEL", "INFO"))                # same thing, shorter

try:
    os.environ["APP_MODE"]
except KeyError as e:
    print(f"{type(e).__name__}: {e}")

Values are always **strings**. Parse numbers and booleans yourself, and be explicit about what counts as "true":

In [ ]:
def env_bool(name: str, default: bool = False) -> bool:
    value = os.environ.get(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "on"}

def env_int(name: str, default: int) -> int:
    value = os.environ.get(name)
    return default if value is None else int(value)

print(env_bool("FEATURE_X_ENABLED"), env_int("WORKER_COUNT", 4))   # False 4 — not set

⚠️ **Gotcha:** on Windows, environment variable names are **case-insensitive** (`os.environ["path"]` works there); on Linux and macOS they are case-sensitive. Use the canonical upper-case names to stay portable.

## Never print the whole environment

It is tempting to write `print(os.environ)` or `for k, v in os.environ.items(): print(k, v)` to "see what's there". Don't — especially in notebooks, logs or CI output:

- environment variables routinely hold **secrets**: API keys, database passwords, cloud credentials, access tokens;
- notebooks are saved *with their output* and get committed, shared and published;
- logs are shipped to systems that many people can read.

Look up only the specific names you need, or print counts and names you have explicitly allowed:

In [ ]:
print(len(os.environ) > 0)                              # how many is machine-specific

safe_to_show = ["LANG", "TZ", "APP_MODE"]               # an explicit allow-list
for name in safe_to_show:
    print(f"{name:9} set: {name in os.environ}")        # whether it is set, not its value

**Tip:** if you must log configuration, log *which* settings are present and mask the values (`DB_PASSWORD=****`).

## Setting variables — and child processes

Assigning to `os.environ` changes the environment of the current process **and every child process it starts afterwards**. It never affects the parent shell that launched Python (the same is true in .NET).

```csharp
// C#
Environment.SetEnvironmentVariable("APP_MODE", "demo");
```

In [ ]:
import subprocess
import sys

os.environ["APP_MODE"] = "demo"          # values must be str
child = subprocess.run([sys.executable, "-c", "import os; print(os.environ.get('APP_MODE'))"],
                       capture_output=True, text=True)
print("child sees:", child.stdout.strip())     # demo

del os.environ["APP_MODE"]               # remove it again (C#: SetEnvironmentVariable(name, null))
print(os.environ.get("APP_MODE"))        # None

⚠️ **Gotcha:** only strings are allowed, so `os.environ["WORKERS"] = 4` raises `TypeError`:

In [ ]:
try:
    os.environ["WORKERS"] = 4
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

Often you only want to change the environment **for one child process**. Pass a modified *copy* via `env=` instead of mutating `os.environ`:

In [ ]:
child_env = {**os.environ, "APP_MODE": "test"}        # copy + override
child = subprocess.run([sys.executable, "-c", "import os; print(os.environ['APP_MODE'])"],
                       capture_output=True, text=True, env=child_env)
print("child sees:", child.stdout.strip())     # test
print("we still see:", os.environ.get("APP_MODE"))   # None — our environment is untouched

For tests, `unittest.mock.patch.dict(os.environ, {...})` temporarily changes variables and restores them afterwards *(see Standard Library › Testing & Debugging › 002 · Mocking with unittest.mock)*:

In [ ]:
from unittest import mock

with mock.patch.dict(os.environ, {"FEATURE_X_ENABLED": "yes"}):
    print(env_bool("FEATURE_X_ENABLED"))     # True inside the block
print(env_bool("FEATURE_X_ENABLED"))         # False — restored

⚠️ **Gotcha:** `os.environ` is read **once** at start-up. The low-level `os.putenv()` / `os.unsetenv()` change the real process environment but do *not* update `os.environ`, so the two can disagree. Always go through `os.environ`.

## Processor count and process id

```csharp
// C#
int cores = Environment.ProcessorCount;
int pid = Environment.ProcessId;
```

In [ ]:
print(os.cpu_count() >= 1)            # logical CPUs in the machine (may be None if unknown!)
print(os.process_cpu_count() >= 1)    # CPUs this process may use (3.13+), respects affinity
print(isinstance(os.getpid(), int))   # this process's id — a different number every run

**Tip:** when sizing a worker pool, prefer `os.process_cpu_count()` (Python 3.13+): in containers or with CPU affinity, a process may be allowed fewer cores than the machine has. That is what .NET's `Environment.ProcessorCount` reports too.

## Platform information with `platform`

The `platform` module answers "what OS / CPU / Python is this?". The values below depend on the machine running the notebook:

```csharp
// C#
Console.WriteLine(RuntimeInformation.OSDescription);
Console.WriteLine(RuntimeInformation.ProcessArchitecture);   // X64, Arm64...
Console.WriteLine(RuntimeInformation.FrameworkDescription);
```

In [ ]:
import platform

print(platform.system())                 # e.g. 'Linux', 'Darwin' (macOS), 'Windows'
print(platform.machine())                # e.g. 'x86_64', 'arm64', 'AMD64'
print(platform.python_version())         # e.g. '3.14.0'
print(platform.python_implementation())  # 'CPython'
print(platform.platform(terse=True))     # short OS description, e.g. 'macOS-15.5'

⚠️ **Gotcha:** `platform.machine()` uses each OS's own naming — the same 64-bit Intel CPU is `x86_64` on Linux/macOS but `AMD64` on Windows, and ARM is `arm64` or `aarch64`. Normalise before comparing:

In [ ]:
def arch() -> str:
    machine = platform.machine().lower()
    return {"amd64": "x64", "x86_64": "x64", "arm64": "arm64", "aarch64": "arm64"}.get(machine, machine)

print(arch() in {"x64", "arm64", platform.machine().lower()})   # True

`platform.node()` (the host name, like `Environment.MachineName`) and `platform.uname()` also exist. Host and user names identify machines and people, so the same "don't dump it into shared output" advice applies.

## OS checks: `sys.platform` vs `platform.system()`

For branching code, the idiomatic check is `sys.platform` — it is a constant, it is fast, and type checkers (mypy, pyright) understand it and skip the code for other platforms. `os.name` is the coarser `"posix"` / `"nt"` split.

| OS | `sys.platform` | `platform.system()` | `os.name` |
|---|---|---|---|
| Windows | `"win32"` | `"Windows"` | `"nt"` |
| Linux | `"linux"` | `"Linux"` | `"posix"` |
| macOS | `"darwin"` | `"Darwin"` | `"posix"` |

```csharp
// C#
if (OperatingSystem.IsWindows()) { ... }
else if (OperatingSystem.IsLinux()) { ... }
```

In [ ]:
if sys.platform == "win32":
    config_hint = "%APPDATA%"
elif sys.platform == "darwin":
    config_hint = "~/Library/Application Support"
else:                                       # linux and other Unix-likes
    config_hint = "~/.config"

print(os.name in {"posix", "nt"}, sys.platform in {"win32", "darwin"} or sys.platform.startswith("linux"))
print("config lives under something like", config_hint)

## Separators: `os.sep`, `os.pathsep`, `os.linesep`

| Constant | Windows | Linux / macOS | .NET |
|---|---|---|---|
| `os.sep` | `\` | `/` | `Path.DirectorySeparatorChar` |
| `os.pathsep` (in `PATH`) | `;` | `:` | `Path.PathSeparator` |
| `os.linesep` | `\r\n` | `\n` | `Environment.NewLine` |

In [ ]:
print(repr(os.sep), repr(os.pathsep), repr(os.linesep))

path_entries = os.environ["PATH"].split(os.pathsep)   # split PATH portably
print(len(path_entries) > 0)                          # (entries are machine-specific paths)

⚠️ **Gotcha:** you rarely need these directly. Build paths with `pathlib` *(see Standard Library › Files & Paths › 001 · Paths with pathlib)*, and **do not** write `os.linesep` to files opened in text mode — Python already translates `"\n"` to the platform's line ending there, so you would get `\r\r\n` on Windows. Just write `"\n"`.

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Read a variable | `Environment.GetEnvironmentVariable("X")` | `os.environ.get("X")` / `os.getenv("X")` |
| Read with default | `... ?? "default"` | `os.environ.get("X", "default")` |
| Required variable | check for `null` and throw | `os.environ["X"]` (raises `KeyError`) |
| Set a variable | `Environment.SetEnvironmentVariable("X", "v")` | `os.environ["X"] = "v"` |
| Remove a variable | `SetEnvironmentVariable("X", null)` | `del os.environ["X"]` / `os.environ.pop("X", None)` |
| Env for one child | `ProcessStartInfo.Environment["X"] = "v"` | `subprocess.run(..., env={**os.environ, "X": "v"})` |
| Temporary change in tests | custom fixture | `mock.patch.dict(os.environ, {...})` |
| CPU count | `Environment.ProcessorCount` | `os.process_cpu_count()` / `os.cpu_count()` |
| Process id | `Environment.ProcessId` | `os.getpid()` |
| Is Windows? | `OperatingSystem.IsWindows()` | `sys.platform == "win32"` |
| OS name | `RuntimeInformation.OSDescription` | `platform.system()` / `platform.platform()` |
| CPU architecture | `RuntimeInformation.ProcessArchitecture` | `platform.machine()` |
| Path separator | `Path.DirectorySeparatorChar` | `os.sep` |
| New line | `Environment.NewLine` | `os.linesep` (but write `"\n"` in text mode) |

## Exercises

1. Write `require_env(name: str) -> str` that returns the variable's value or raises a `RuntimeError` with a helpful message. Show both outcomes using `mock.patch.dict`.
2. Write `masked_settings(names: list[str]) -> dict[str, str]` that returns, for each name, `"<not set>"` if missing, the value if the name does **not** contain `KEY`, `SECRET`, `TOKEN` or `PASSWORD`, and `"****"` otherwise. Test it with a patched environment containing `APP_MODE=demo` and `API_TOKEN=fake-token-123`.
3. Start a child Python process with an environment that contains **only** `{"GREETING": "hello"}` plus whatever is needed to run (hint: on Windows `SYSTEMROOT` is required — copy it if present) and print what the child sees for `GREETING`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import os
import subprocess
import sys
from unittest import mock

# 1 — a clear error beats a bare KeyError deep in the stack
def require_env(name: str) -> str:
    try:
        return os.environ[name]
    except KeyError:
        raise RuntimeError(f"required environment variable {name!r} is not set") from None

with mock.patch.dict(os.environ, {"DB_HOST": "localhost"}):
    print(require_env("DB_HOST"))
try:
    require_env("DB_HOST_MISSING")
except RuntimeError as e:
    print(f"{type(e).__name__}: {e}")

# 2 — mask anything that looks secret
SECRET_MARKERS = ("KEY", "SECRET", "TOKEN", "PASSWORD")

def masked_settings(names: list[str]) -> dict[str, str]:
    result = {}
    for name in names:
        value = os.environ.get(name)
        if value is None:
            result[name] = "<not set>"
        elif any(marker in name.upper() for marker in SECRET_MARKERS):
            result[name] = "****"
        else:
            result[name] = value
    return result

with mock.patch.dict(os.environ, {"APP_MODE": "demo", "API_TOKEN": "fake-token-123"}):
    print(masked_settings(["APP_MODE", "API_TOKEN", "LOG_DIR_UNSET"]))

# 3 — env= REPLACES the whole environment, so build it deliberately
minimal_env = {"GREETING": "hello"}
if "SYSTEMROOT" in os.environ:
    minimal_env["SYSTEMROOT"] = os.environ["SYSTEMROOT"]
child = subprocess.run(
    [sys.executable, "-c", "import os; print(os.environ.get('GREETING'))"],
    capture_output=True, text=True, env=minimal_env,
)
print(child.stdout.strip())     # hello
```

</details>